# Official baseline for "Handwritten to Data"

A two-stage detect-then-transcribe pipeline on **Qwen3-VL-8B-Instruct** served via vLLM (OpenAI-compatible API). Every stage uses `guided_json` (xgrammar backend) for grammar-constrained structured output, orchestrated as an async LangGraph `StateGraph` with a single `asyncio.Semaphore` as the concurrency knob.

## Competition metric

`Score = 0.15 × Detection_F1 + 0.05 × ClassAcc + 0.30 × (1 − Region_CER) + 0.50 × (1 − Page_CER)`

Page CER carries 50 % of the weight, so the pipeline is tuned end-to-end for transcription quality on the concatenated page text.

## Architecture

```
            ┌──────────────┐
            │  load_image  │  loads at ORIGINAL resolution; bbox coords
            └──────┬───────┘  stay in original-image space throughout.
                   │          (encode_jpeg() area-scales the b64 payload
                   │           sent to Qwen — coords come back as 0-1000
                   │           relative and are rescaled to original.)
                   ↓
            ┌─────────────────────────────────────────────────┐
            │ detect (single async node)                      │
            │   • dictation/archive: block_detect → per-block │
            │     line_detect (asyncio.gather, parallel)      │
            │   • school/university: classify_page first      │
            │       – math   → per-line detect+type           │
            │                  (fallback to block→line if ≤2) │
            │       – text   → block_detect → per-block       │
            │                  line_detect (parallel)         │
            │   non-text blocks (table/formula/annotation/    │
            │   image/graph) skip line_detect — single region │
            └──────┬──────────────────────────────────────────┘
                   │ Send(transcribe × N regions)
                   ↓
            ┌─────────────────────────┐
            │ transcribe_region       │ × N parallel
            │ per-type prompt:        │ reducer concatenates
            │ handwritten / printed / │ results into state.regions
            │ formula / table /       │ (image / graph → empty)
            │ annotation              │
            └─────────────────────────┘
```

## Key engineering decisions

1. **Two-stage detect (block → lines on crop)** rather than single-pass. The block prompt enforces _"merge consecutive text lines into ONE text_block"_; the line detector then runs on a tight crop of each block, where lines occupy a much larger fraction of the image → cleaner per-line bboxes, no overlapping duplicates.

2. **`writing_type` as a first-class field in block detect**. Without it, archive documents (1920s typed bodies with handwritten signatures and stamps) get blanket-labelled as handwritten, hurting both classification accuracy and the choice of transcription prompt downstream.

3. **Area-based smart resize** (`MAX_PIXELS = 2048 × 28²`) instead of `max(w, h)`-based. A 4284×5712 page resized only by max-side is still 3M+ pixels — the server then rescales again, doubling work and bloating the b64 payload.

4. **Tight transcribe crop margin (5 %)** — adjacent line bboxes from the line detector often overlap vertically by ~30-50 %; a generous margin bleeds the previous/next line into the crop. The prompt also says explicitly _"transcribe ONLY the centered line; ANY text from the line above or below is OUT OF SCOPE"_.

5. **0-1000 normalized coordinates** throughout (resize-invariant — sidesteps Qwen3-VL's `smart_resize` coordinate drift).

6. **xgrammar `guided_json` schema on every stage** (page-type, blocks, lines, transcribe). 2-5× faster than outlines on nested JSON schemas, and guarantees valid JSON with no post-processing.

7. **Source-aware routing** — school / university pages first hit a page-type classifier (text vs math). If math, a single per-line detect+classify call short-circuits the block→line cascade. Fewer LLM calls on formula-heavy pages.

8. **Digit-list rule** + **no-context-substitution rule** in the transcribe prompt. The model used to confuse `"2."` with `"в."` (Cyrillic) and substitute short isolated words with plausible-sounding alternatives (e.g. `"Франції"` → `"Ранені"`); explicit rules cut this down.

9. **Fully async**: `AsyncOpenAI` + LangGraph `ainvoke`, a single module-level `asyncio.Semaphore(N)` as concurrency knob — no thread pool, no GIL pressure. `tenacity` retries with exponential backoff on transient 429 / 5xx / timeouts.

## Test-set scores

| Component | Score |
|---|---|
| Detection F1 | 0.778 (precision 0.81 / recall 0.75) |
| Classification accuracy | 0.946 |
| Region CER | 0.609 |
| Page CER | 0.576 |
| **Composite** | **0.635** |

## vLLM endpoint (how it was deployed)

Inference goes through an external OpenAI-compatible vLLM server. I ran inference on 1× **NVIDIA A40 (48 GB)** on RunPod with vLLM 0.19.x:

```bash
vllm serve Qwen/Qwen3-VL-8B-Instruct \
  --port 8000 \
  --dtype auto \
  --gpu-memory-utilization 0.92 \
  --max-model-len 16384 \
  --max-num-seqs 16 \
  --limit-mm-per-prompt '{"image":8}' \
  --mm-processor-kwargs '{"min_pixels":401408,"max_pixels":1605632}' \
  --mm-processor-cache-gb 0 \
  --structured-outputs-config '{"backend":"xgrammar"}' \
  --served-model-name Qwen/Qwen3-VL-8B-Instruct \
  --api-key sk-... \
  --trust-remote-code
```

**Why each non-default flag matters:**
- `--mm-processor-cache-gb 0` — workaround for a vLLM 0.19.1 bug where the multimodal cache deadlocks the engine on roughly 1 in 1 000 requests
- `--structured-outputs-config '{"backend":"xgrammar"}'` — 2-5× faster than the default `outlines` backend on nested JSON schemas
- `--mm-processor-kwargs` matches the client-side resize budget so the server doesn't redundantly rescale every image
- `--max-num-seqs 16` enables ~10× real concurrency at full 16k context on this GPU

Set your own `BASE_URL` and `API_KEY` in the configuration cell below.

## Install dependencies

In [ ]:
%pip install --quiet openai langgraph tenacity pillow tqdm typing_extensions huggingface_hub matplotlib

## Configuration

Credentials are loaded via **Kaggle Secrets** (`Add-ons → Secrets` in the Kaggle notebook UI). Add two secrets:

* `QWEN_BASE_URL` — your vLLM endpoint, e.g. `https://YOUR_POD_ID-8000.proxy.runpod.net/v1`
* `QWEN_API_KEY` — the API key you set on the server with `--api-key`

When the notebook is published, the secrets stay private — readers see the code but not the values.

If you're running outside Kaggle (locally or in a different environment), set the same names as environment variables before importing this notebook.

In [ ]:
import os

# Load credentials. Order:
#   1. Kaggle Secrets (preferred — invisible to readers of a published notebook)
#   2. Pre-existing env vars (handy for local development)
#   3. Placeholder fallbacks (so the notebook at least imports)
try:
    from kaggle_secrets import UserSecretsClient
    _ks = UserSecretsClient()
    os.environ["QWEN_BASE_URL"] = _ks.get_secret("QWEN_BASE_URL")
    os.environ["QWEN_API_KEY"]  = _ks.get_secret("QWEN_API_KEY")
    print("Credentials loaded from Kaggle Secrets.")
except Exception:
    os.environ.setdefault("QWEN_BASE_URL", "https://YOUR_POD_ID-8000.proxy.runpod.net/v1")
    os.environ.setdefault("QWEN_API_KEY",  "sk-REPLACE_ME")
    print("Kaggle Secrets unavailable — using env vars / placeholders.")

os.environ.setdefault("QWEN_MODEL", "Qwen/Qwen3-VL-8B-Instruct")

# Concurrency: cap on simultaneous LLM calls and in-flight images.
# Tuned for the vLLM A40 endpoint (max_num_seqs=16, ~10x real parallelism).
# Lower if your endpoint queues; raise if it idles.
CONCURRENCY = 8

# Test data location (adjust to where the competition data is mounted)
TEST_IMAGES_DIR = "/kaggle/input/rukopys/test/images"
TEST_METADATA   = "/kaggle/input/rukopys/test/metadata.jsonl"

# Outputs
OUTPUT_DIR     = "/kaggle/working/predictions"
SUBMISSION_CSV = "/kaggle/working/submission.csv"


## Pipeline

The full implementation — imports, config, async LLM client with tenacity retry, prompts, JSON schemas, parsers, graph nodes, graph assembly, and the per-image runner. Qwen-VL receives ≤2048 image tokens per image; bounding boxes come back in a 0-1000 normalized scale (resize-invariant).

In [ ]:
from __future__ import annotations

import argparse  # noqa: F401  (kept for parity with source pipeline)
import asyncio
import base64
import json
import operator
import os
import re
import threading
import time
from functools import lru_cache
from io import BytesIO
from pathlib import Path
from typing import Annotated, Any

from PIL import Image, ImageDraw, ImageFont, ImageOps  # noqa: F401
from openai import AsyncOpenAI
from tenacity import (
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential,
)
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

# ── Config ─────────────────────────────────────────────────────────────────
#
# OpenAI-compatible Qwen-VL endpoint. Override via env if you have your own
# deployment.
#
# Endpoint constraints (runpod A40 46 GB, vLLM BF16, xgrammar guided_json):
#   * max_model_len = 16384 tokens (prompt + completion combined)
#   * max_num_seqs = 16 — real parallelism ≈ 10× at full 16k context
#   * server-side image limits: min_pixels=401408, max_pixels=1605632
#     (matched to client MIN_PIXELS/MAX_PIXELS — no server-side rescale)
#   * reasoning parser OFF (Instruct, not Thinking), tool parser OFF
# These drive MAX_PIXELS and the default ``--concurrency`` value below.

VLLM_URL = os.environ["QWEN_BASE_URL"]   # set in the Configuration cell
VLLM_KEY = os.environ["QWEN_API_KEY"]    # set in the Configuration cell
QWEN_MODEL = os.getenv("QWEN_MODEL", "Qwen/Qwen3-VL-8B-Instruct")

# Image token budget. For Qwen3-VL: tokens ≈ pixels / (28*28).
# 2048*28*28 → ≤2048 image tokens; with ~500 prompt tokens + ~4000 completion
# this fits within 16384 tokens with ~9k headroom for dense detection
# responses (~50+ regions). Aligned with server-side max_pixels=1605632.
MAX_PIXELS = 2048 * 28 * 28
MIN_PIXELS = 512 * 28 * 28

# Assume every page is Ukrainian — no per-block language classification.
DEFAULT_LANGUAGE = "uk"


# ── Thread-safe logging ────────────────────────────────────────────────────

_print_lock = threading.Lock()


def _log(*args, **kwargs):
    with _print_lock:
        print(*args, **kwargs, flush=True)


# ── Async client + concurrency gate ────────────────────────────────────────

@lru_cache(maxsize=1)
def _qwen_client() -> AsyncOpenAI:
    # 180s timeout per request. A single Qwen-VL call usually takes 5-30s on
    # A40, but server-side queueing (max_num_seqs=16) can push p99 latency to
    # 60-120s under load. Without an explicit timeout the SDK defaults to 600s
    # and a single hung TCP connection wedges the whole batch for up to 40 min
    # (4 × 600s with tenacity retries). max_retries=0 lets tenacity own the
    # retry policy end-to-end.
    return AsyncOpenAI(api_key=VLLM_KEY, base_url=VLLM_URL,
                       timeout=180.0, max_retries=0)


# A single semaphore bounds in-flight vLLM requests across the whole process.
# Initialised by ``init_concurrency`` (or implicitly with a safe default the
# first time it is read). One knob: ``--concurrency``.
# Default tuned for the runpod A40 endpoint (max_num_seqs=16, ~10× real
# parallelism at 16k context). Lower this if hitting OOM or queueing.
_QWEN_SEM: asyncio.Semaphore | None = None
_DEFAULT_CONCURRENCY = 8


def init_concurrency(n: int) -> None:
    """Set the global vLLM concurrency limit. Must be called inside a running loop."""
    global _QWEN_SEM
    _QWEN_SEM = asyncio.Semaphore(max(1, n))


def _qwen_sem() -> asyncio.Semaphore:
    global _QWEN_SEM
    if _QWEN_SEM is None:
        _QWEN_SEM = asyncio.Semaphore(_DEFAULT_CONCURRENCY)
    return _QWEN_SEM


# ── LLM call ───────────────────────────────────────────────────────────────

_TRANSIENT_MARKERS = ("429", "500", "502", "503", "504", "timeout", "timed out",
                      "connection", "connect", "rate limit")


def _is_transient(exc: BaseException) -> bool:
    """Retry on rate limits, server errors, and connection blips. Fail fast on the rest."""
    msg = str(exc).lower()
    return any(m in msg for m in _TRANSIENT_MARKERS)


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1.5, min=1, max=20),
    retry=retry_if_exception(_is_transient),
    reraise=True,
)
async def _qwen_chat(image_b64: str, prompt: str, schema: dict | None,
                     max_tokens: int) -> str:
    """Single Qwen-VL request, gated by ``_QWEN_SEM`` and tenacity-retried."""
    extra_body: dict[str, Any] = {
        "chat_template_kwargs": {"enable_thinking": False},
        "mm_processor_kwargs": {"max_pixels": MAX_PIXELS, "min_pixels": MIN_PIXELS},
    }
    if schema is not None:
        extra_body["guided_json"] = schema

    async with _qwen_sem():
        resp = await _qwen_client().chat.completions.create(
            model=QWEN_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
                {"type": "text", "text": prompt},
            ]}],
            max_tokens=max_tokens, temperature=0,
            extra_body=extra_body,
        )
    raw = resp.choices[0].message.content or ""
    # guided_json gives valid JSON; strip ``<think>`` and code fences defensively.
    raw = re.sub(r"<think>[\s\S]*?</think>", "", raw).strip()
    raw = re.sub(r"```json\s*", "", raw)
    raw = re.sub(r"```\s*$", "", raw).strip()
    return raw


async def qwen_call(image_b64: str, prompt: str, *,
                    schema: dict | None = None,
                    max_tokens: int = 4000) -> str:
    """Call Qwen-VL via vLLM. Never raises — returns ``[ERROR:*]`` on failure."""
    try:
        return await _qwen_chat(image_b64, prompt, schema, max_tokens)
    except Exception as e:
        _log(f"    QWEN failed: {str(e)[:120]}")
        return "[ERROR:qwen_failed]"


# ── Image helpers ──────────────────────────────────────────────────────────

def _resize_for_qwen(img: Image.Image, max_pixels: int = MAX_PIXELS) -> Image.Image:
    """Resize so total pixel count ≤ ``max_pixels`` (matches server-side limit).

    Resizing by max(w,h) alone is insufficient: a 4284×5712 page clamped to
    max(w,h)=2048 still has 3.1M pixels — 2× the server limit (1.6M). The
    server then rescales again, doubling work and inflating the b64 payload.
    Scaling by area ensures the image arrives at exactly the server's image-
    token budget — no wasted bytes, no wasted compute.
    """
    w, h = img.size
    pixels = w * h
    if pixels <= max_pixels:
        return img
    scale = (max_pixels / pixels) ** 0.5
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def encode_jpeg(img: Image.Image) -> bytes:
    buf = BytesIO()
    _resize_for_qwen(img).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


# ── JSON parsing (simple — vLLM guided_json gives us valid JSON) ───────────

def parse_json(raw: str) -> list | dict | None:
    if not raw or raw.startswith("[ERROR:"):
        return None
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"[\[\{][\s\S]*[\]\}]", raw)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                return None
        return None


# ── Prompts ────────────────────────────────────────────────────────────────

# Stage 0: page classifier — text-heavy vs math-heavy. Used only for school /
# university pages. Drives the math short-circuit (skip block detect, go per-line).
QWEN_CLASSIFY_PAGE_PROMPT = """Look at this school/student page. What type of content dominates?
Return JSON: {"page_type": "text"} or {"page_type": "math"}
- "text"  — mostly handwritten text (language, literature, history, biology,
            geography, social studies, essays)
- "math"  — formulas, equations, numbers, calculations, tables (math, physics,
            chemistry problems)
ONLY valid JSON."""


# Stage 1 (text path): block-level detection on the FULL page. Strict
# merge rules → one bbox per paragraph, not per line.
QWEN_BLOCK_DETECT_PROMPT = """Detect content BLOCKS in this document image.

block_type:
- "text_block"  (paragraph of text lines)
- "table"       (rows + columns)
- "formula"     (standalone math/chemistry equation on its OWN line, NOT inline)
- "image"       (stamp / seal / drawing)
- "graph"       (chart / diagram with axes)
- "annotation"  (grade / date / page number / exercise number / teacher mark / signature)

writing_type (per block):
- "handwritten" — written by hand
- "printed"     — typed / typeset / machine-printed / stamped

Rules:
- Merge consecutive text lines into ONE text_block
- Full-page text = ONE text_block
- Tables, annotations = ALWAYS separate blocks
- Formula = separate block ONLY if it stands alone on its own line(s)
- Text above/below a table = separate text_blocks
- A page can MIX handwritten and printed blocks (typed body + handwritten
  signature/note). Classify writing_type per block, not per page.
- Skip vertical text (rotated > 45°)
- Ignore ruled/grid/squared background lines — they are NOT text

Return: {"blocks": [{"bbox_2d": [x1,y1,x2,y2], "block_type": "text_block", "writing_type": "handwritten"}]}
Coordinates 0-1000 scale. ONLY valid JSON."""


# Stage 2 (text path): line-level detection within a CROP of one text_block.
# No type classification needed — we already know it's text.
QWEN_LINE_DETECT_PROMPT = """Detect each TEXT LINE in this image (one visual row = one line).
Return: {"lines": [{"bbox_2d": [x1,y1,x2,y2]}]}
Coordinates 0-1000 scale, top-to-bottom order. ONLY valid JSON."""


# Math short-circuit: per-line detect + classify in ONE call on the full page.
# School variant.
QWEN_SCHOOL_PERLINE_PROMPT = """Detect every LINE in this school notebook page (grades 5-11, various subjects).

Each line = one visual row. Do NOT merge multiple lines.
Ignore ruled/grid lines in the background — they are NOT text.

For each line, classify:
- "formula"     — line contains math/chemistry: equations (=), variables,
                  fractions, chemical formulas
- "text_block"  — plain text (Ukrainian language, literature, geography,
                  history, etc.)
- "annotation"  — date, page number, exercise number (e.g. "Вправа 430"),
                  grade, teacher mark
- "table"       — ENTIRE vertical calculation block with a dividing line.
                  ONE bbox covering ALL rows.

Return: {"blocks": [{"bbox_2d": [x1,y1,x2,y2], "block_type": "text_block"}]}
Coordinates 0-1000 scale. ONLY valid JSON."""


# Math short-circuit: per-line detect + classify. University variant.
QWEN_UNIVERSITY_PERLINE_PROMPT = """Detect every LINE in this university student work (math, formulas, text).

Each line = one visual row. Do NOT merge multiple lines.

For each line, classify:
- "formula"     — line contains math symbols, equations, set notation,
                  matrices, variables, inequalities
- "text_block"  — plain text without math
- "annotation"  — grade, date, task number
- "image"       — drawing, diagram without axes
- "graph"       — chart with coordinate axes
- "table"       — table row

Return: {"blocks": [{"bbox_2d": [x1,y1,x2,y2], "block_type": "formula"}]}
Coordinates 0-1000 scale. ONLY valid JSON."""


QWEN_TRANSCRIBE_PROMPT = """Transcribe ONLY the text physically written in this crop. Read left to right.

RULES:
- This crop contains ONE main line of text. Transcribe ONLY that central line.
  ANY text from the line above or below — even if it looks fully written — is OUT OF SCOPE. Do NOT include it. Output the central line only.
- If you cannot read a word, write the LETTERS you see — do NOT substitute a similar real word from context.
- Numbered list lines start with a DIGIT + period ("1.", "2.", "3.", "10."). Never a Cyrillic letter (а., в.) when it is clearly a digit.
- Ukrainian Cyrillic ONLY, never Latin. These glyphs are Cyrillic, not Latin: В Н С Р Т К Е О А М Х.
- Edge cut-off: visible part with hyphen ("вирі-"). Crossed-out: ~~закреслено~~ or ~~old~~{new}. No memory completion.
- Formulas: plain text with spacing ("2x + 3 = 7", "H₂SO₄"). Tables: cells separated by |.

Return JSON: {"text": "..."}"""

QWEN_FORMULA_PROMPT = """Transcribe this handwritten math/chemistry formula exactly as written.

Examples of correct transcription:
- "P(A) = 1/60 · 36 ? P(B) = 1/10"
- "x - 41 = 0"
- "|-57| · |x| = 0"
- "R₁ ∩ R₂ = {(a, b)}"
- "TC(Q) = 4Q^2 + 40Q + 400"
- "CH_3-COOH + NaOH → CH_3COONa + H_2O"
- "НСК (56, 35) = 2 * 2 * 2 * 7 * 5 = 280"
- "\\frac{d}{dx} f(x) = \\lim_{h \\to 0} \\frac{f(x+h) - f(x)}{h}"
- "\\sqrt{16} = 4"
- "S = 4 · 0,000 03 = 0,00012 м^2"

Rules:
- Use LaTeX for fractions (\\frac{}{}), roots (\\sqrt{}), limits (\\lim), arrows (\\to)
- Use ^{} for superscripts, _{} for subscripts: x^{2}, H_{2}O
- Simple cases: x^2, x_3 (no braces for single char)
- Ukrainian Cyrillic for text (В ≠ B, С ≠ C, Н ≠ H)
- Preserve exact notation as written

Return JSON: {"text": "..."}"""

QWEN_TABLE_PROMPT = ('Transcribe this table. Each row on a new line, cells separated by |. '
                    'Return JSON: {"text": "..."}')

QWEN_ANNOTATION_PROMPT = ('Transcribe this annotation (grade / date / page number / exercise number). '
                         'Return JSON: {"text": "..."}')


# ── JSON schemas for vLLM guided_json structured output ────────────────────

BLOCK_TYPES = ["text_block", "table", "formula", "image", "graph", "annotation"]

_BBOX_SCHEMA = {
    "type": "array", "items": {"type": "integer"},
    "minItems": 4, "maxItems": 4,
}

PAGE_TYPE_SCHEMA = {
    "type": "object",
    "properties": {"page_type": {"type": "string", "enum": ["text", "math"]}},
    "required": ["page_type"],
}

BLOCKS_SCHEMA = {
    "type": "object",
    "properties": {
        "blocks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "bbox_2d":      _BBOX_SCHEMA,
                    "block_type":   {"type": "string", "enum": BLOCK_TYPES},
                    "writing_type": {"type": "string",
                                     "enum": ["handwritten", "printed"]},
                },
                "required": ["bbox_2d", "block_type", "writing_type"],
            },
        },
    },
    "required": ["blocks"],
}

# Line-detect output: just bboxes — block_type is implicit (text_block,
# since we only call this on a text_block crop).
LINES_SCHEMA = {
    "type": "object",
    "properties": {
        "lines": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"bbox_2d": _BBOX_SCHEMA},
                "required": ["bbox_2d"],
            },
        },
    },
    "required": ["lines"],
}

# Per-line schema (math short-circuit): bbox + block_type per line.
# Output key is "blocks" to match the legacy contract — the prompt uses
# the same wording, and existing parsers expect this name.
PERLINE_SCHEMA = BLOCKS_SCHEMA

TRANSCRIBE_SCHEMA = {
    "type": "object",
    "properties": {"text": {"type": "string"}},
    "required": ["text"],
}


# ── State schemas ──────────────────────────────────────────────────────────

class RegionTask(TypedDict):
    """Detected region awaiting transcription."""
    bbox: list[int]
    rtype: str       # handwritten|printed|formula|table|annotation|image|graph
    legibility: str  # legible|illegible


class Region(TypedDict):
    """Final annotated region."""
    type: str
    bbox: list[int]
    language: str       # always DEFAULT_LANGUAGE
    legibility: str
    transcription: str


class PipelineState(TypedDict, total=False):
    # Input
    uuid: str
    source: str         # dictation|archive|school|university
    image_path: str
    # Loaded artifacts (populated by load_image)
    img: Any            # PIL.Image
    img_w: int
    img_h: int
    img_b64: str        # for Qwen
    # Accumulators (reducer-merged across parallel workers)
    tasks: Annotated[list[RegionTask], operator.add]
    regions: Annotated[list[Region], operator.add]


class TranscribeInput(TypedDict):
    """Per-region worker state (dispatched via Send)."""
    task: RegionTask
    img: Any
    img_w: int
    img_h: int


# ── Parsers ────────────────────────────────────────────────────────────────

def _scale_bbox(bbox: list, w: int, h: int,
                off_x: int = 0, off_y: int = 0) -> list[int] | None:
    """Convert a 0-1000 bbox into pixel coords (with optional offset).

    Returns ``None`` for malformed or degenerate boxes (<3 px in either axis).
    """
    if len(bbox) < 4:
        return None
    x1 = int(bbox[0] * w / 1000) + off_x
    y1 = int(bbox[1] * h / 1000) + off_y
    x2 = int(bbox[2] * w / 1000) + off_x
    y2 = int(bbox[3] * h / 1000) + off_y
    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [x1, y1, x2, y2]


def _items_from_payload(raw: str, *keys: str) -> list:
    """Pull a list out of Qwen JSON, tolerating bare-array outputs.

    The runpod Qwen-VL build occasionally drops the root object even with
    guided_json, returning ``[...]`` instead of ``{"<key>": [...]}``.
    """
    data = parse_json(raw)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for k in keys:
            if k in data:
                return data[k] or []
        return []
    return []


def _parse_blocks(raw: str, img_w: int, img_h: int) -> list[dict]:
    """Block-detect output → list of dicts with bbox, block_type, writing_type."""
    blocks = []
    for item in _items_from_payload(raw, "blocks", "regions"):
        bbox = item.get("bbox_2d", item.get("bbox", []))
        scaled = _scale_bbox(bbox, img_w, img_h)
        if scaled is None:
            continue
        blocks.append({
            "bbox": scaled,
            "block_type":   item.get("block_type", "text_block"),
            "writing_type": item.get("writing_type", "handwritten"),
        })
    return blocks


def _parse_lines(raw: str, crop_w: int, crop_h: int,
                 off_x: int, off_y: int) -> list[list[int]]:
    """Line-detect output → list of pixel bboxes in GLOBAL coords."""
    out: list[list[int]] = []
    for item in _items_from_payload(raw, "lines", "blocks"):
        bbox = item.get("bbox_2d", item.get("bbox", item)) if isinstance(item, dict) else item
        scaled = _scale_bbox(bbox, crop_w, crop_h, off_x, off_y)
        if scaled is not None:
            out.append(scaled)
    return out


def _parse_perline(raw: str, img_w: int, img_h: int) -> list[RegionTask]:
    """Per-line detect output (math path) → ready-to-transcribe ``RegionTask``s.

    For ``text_block``-classified lines we set ``rtype="handwritten"`` so the
    transcription prompt matches; for everything else ``rtype`` is the
    declared block_type.
    """
    tasks: list[RegionTask] = []
    for item in _items_from_payload(raw, "blocks", "lines", "regions"):
        bbox = item.get("bbox_2d", item.get("bbox", []))
        scaled = _scale_bbox(bbox, img_w, img_h)
        if scaled is None:
            continue
        block_type = item.get("block_type", "text_block")
        rtype = "handwritten" if block_type == "text_block" else block_type
        tasks.append({"bbox": scaled, "rtype": rtype, "legibility": "legible"})
    return tasks


_TRANSCRIBE_VARIANTS = {
    "formula":    (QWEN_FORMULA_PROMPT,    1024),
    "table":      (QWEN_TABLE_PROMPT,      1024),
    "annotation": (QWEN_ANNOTATION_PROMPT,  512),
}


async def _crop_transcribe(img: Image.Image, bbox: list[int], rtype: str) -> str:
    """Crop a region and transcribe it with Qwen-VL (guided JSON).

    Vertical margin is intentionally tiny (5%): adjacent line bboxes from
    the line detector frequently overlap by 20-40 px, and a generous margin
    bleeds the previous/next line into the crop. The prompt also tells the
    model to ignore intruding letter-tops/bottoms — but the simplest defence
    is to not feed them in the first place.
    """
    img_w, img_h = img.size
    x1, y1, x2, y2 = bbox
    mx, my = int((x2 - x1) * 0.05), int((y2 - y1) * 0.05)
    crop = img.crop((max(0, x1 - mx), max(0, y1 - my),
                     min(img_w, x2 + mx), min(img_h, y2 + my)))
    cw, ch = crop.size
    if max(cw, ch) < 128:
        scale = 256 / max(cw, ch)
        crop = crop.resize((int(cw * scale), int(ch * scale)), Image.LANCZOS)
    crop_b64 = base64.b64encode(encode_jpeg(crop)).decode()

    prompt, max_tok = _TRANSCRIBE_VARIANTS.get(rtype, (QWEN_TRANSCRIBE_PROMPT, 1024))
    raw = await qwen_call(crop_b64, prompt, schema=TRANSCRIBE_SCHEMA, max_tokens=max_tok)
    if raw.startswith("[ERROR:"):
        return raw
    parsed = parse_json(raw)
    if parsed and isinstance(parsed, dict):
        return parsed.get("text", "") or ""
    return "[ERROR:parse_failed]"


# ── Nodes ──────────────────────────────────────────────────────────────────

def load_image(state: PipelineState) -> dict:
    """Entry node: load image at ORIGINAL resolution; bbox coords are in
    original-image space throughout the pipeline.

    The base64 payload sent to Qwen is downsampled internally by
    ``encode_jpeg`` (max_size=2048) for bandwidth, but Qwen returns 0-1000
    relative coords that we re-scale against the ORIGINAL ``img_w``/``img_h``.
    This way the saved bboxes line up with the source file — and the
    ``visualize`` step (which opens the original) draws correctly.
    """
    img = ImageOps.exif_transpose(Image.open(state["image_path"])).convert("RGB")
    img_w, img_h = img.size
    img_b64 = base64.b64encode(encode_jpeg(img)).decode()
    _log(f"\n  {state['source']}/{state['uuid'][:20]}  {img_w}×{img_h}")
    return {"img": img, "img_w": img_w, "img_h": img_h, "img_b64": img_b64}


async def _classify_page(b64: str) -> str:
    """Returns "text" or "math"; defaults to "text" on parse failure."""
    raw = await qwen_call(b64, QWEN_CLASSIFY_PAGE_PROMPT,
                          schema=PAGE_TYPE_SCHEMA, max_tokens=64)
    parsed = parse_json(raw)
    if isinstance(parsed, dict):
        return "math" if parsed.get("page_type") == "math" else "text"
    return "text"


async def _detect_perline(b64: str, source: str,
                          img_w: int, img_h: int) -> list[RegionTask]:
    """Math short-circuit: one call → per-line bboxes WITH block_type."""
    prompt = (QWEN_UNIVERSITY_PERLINE_PROMPT if source == "university"
              else QWEN_SCHOOL_PERLINE_PROMPT)
    raw = await qwen_call(b64, prompt, schema=PERLINE_SCHEMA)
    return _parse_perline(raw, img_w, img_h)


async def _lines_in_block(img: Image.Image, block: dict) -> list[RegionTask]:
    """Crop one text_block, ask Qwen for per-line bboxes, lift to global coords.

    Non-text block types (image/graph/table/formula/annotation) bypass the
    line-detect call and are returned as a single task with ``rtype`` set to
    the block_type — table/formula/annotation get whole-block transcription;
    image/graph get an empty transcription downstream.
    """
    btype = block["block_type"]
    bbox = block["bbox"]

    if btype == "text_block":
        # writing_type comes from block detect (handwritten or printed) and
        # propagates to every line in this block — a typed paragraph stays
        # typed, a hand-written one stays hand-written.
        wtype = block.get("writing_type", "handwritten")
        img_w, img_h = img.size
        x1, y1, x2, y2 = bbox
        # ~3% margin so the line detector sees the full first/last line.
        mx, my = int((x2 - x1) * 0.03), int((y2 - y1) * 0.03)
        cx1, cy1 = max(0, x1 - mx), max(0, y1 - my)
        cx2, cy2 = min(img_w, x2 + mx), min(img_h, y2 + my)
        crop = img.crop((cx1, cy1, cx2, cy2))
        crop_w, crop_h = crop.size
        crop_b64 = base64.b64encode(encode_jpeg(crop)).decode()
        raw = await qwen_call(crop_b64, QWEN_LINE_DETECT_PROMPT, schema=LINES_SCHEMA)
        lines = _parse_lines(raw, crop_w, crop_h, cx1, cy1)
        if not lines:  # fallback: keep block as one region
            return [{"bbox": bbox, "rtype": wtype, "legibility": "legible"}]
        return [{"bbox": lb, "rtype": wtype, "legibility": "legible"}
                for lb in lines]

    # Non-text blocks: keep as a single bbox, downstream picks the right
    # transcription prompt based on rtype (or skips for image/graph).
    return [{"bbox": bbox, "rtype": btype, "legibility": "legible"}]


async def detect(state: PipelineState) -> dict:
    """Two-stage source-aware detection.

    For school/university: classify text vs math first.
        * math → single per-line call (bbox + block_type per line)
        * text → block detect on full page → per-block parallel line detect

    For dictation/archive: always block → per-block line detect.

    A failed math short-circuit (≤2 lines) falls back to the text path so a
    single bad page-type guess does not waste the whole document.
    """
    img = state["img"]
    img_w, img_h = state["img_w"], state["img_h"]
    b64 = state["img_b64"]
    source = state["source"]

    if source in ("school", "university"):
        page_type = await _classify_page(b64)
        _log(f"    page type: {page_type}")
        if page_type == "math":
            tasks = await _detect_perline(b64, source, img_w, img_h)
            if len(tasks) > 2:
                _log(f"    per-line: {len(tasks)} lines")
                return {"tasks": tasks}
            _log(f"    per-line yielded {len(tasks)}, falling back to block→line")

    raw = await qwen_call(b64, QWEN_BLOCK_DETECT_PROMPT, schema=BLOCKS_SCHEMA)
    blocks = _parse_blocks(raw, img_w, img_h)
    _log(f"    blocks: {len(blocks)}")
    # Per-block line detect runs concurrently — bounded by the global
    # _QWEN_SEM, so we don't need to throttle here.
    per_block = await asyncio.gather(*(_lines_in_block(img, b) for b in blocks))
    tasks = [t for sub in per_block for t in sub]
    _log(f"    regions: {len(tasks)}")
    return {"tasks": tasks}


def fan_out_transcribe(state: PipelineState) -> list[Send]:
    """Emit one Send per task → parallel transcribe_region workers."""
    return [
        Send("transcribe_region", {
            "task": task,
            "img": state["img"],
            "img_w": state["img_w"],
            "img_h": state["img_h"],
        })
        for task in state.get("tasks", [])
    ]


async def transcribe_region(state: TranscribeInput) -> dict:
    """Worker: transcribe one region; reducer merges into ``state.regions``."""
    t = state["task"]
    if t["rtype"] in ("image", "graph") or t["legibility"] == "illegible":
        text = ""
    else:
        text = await _crop_transcribe(state["img"], t["bbox"], t["rtype"])
        text = (text or "").split("\n")[0].strip()
    region: Region = {
        "type": t["rtype"], "bbox": t["bbox"],
        "language": DEFAULT_LANGUAGE, "legibility": t["legibility"],
        "transcription": text,
    }
    return {"regions": [region]}


# ── Graph assembly ─────────────────────────────────────────────────────────

def build_graph():
    g = StateGraph(PipelineState)

    g.add_node("load_image", load_image)
    g.add_node("detect", detect)
    g.add_node("transcribe_region", transcribe_region)

    g.add_edge(START, "load_image")
    g.add_edge("load_image", "detect")
    g.add_conditional_edges("detect", fan_out_transcribe, ["transcribe_region"])
    g.add_edge("transcribe_region", END)

    return g.compile()


# ── Per-image runner ───────────────────────────────────────────────────────

async def run_pipeline(graph, image_path: str, uuid: str, source: str) -> dict:
    """Invoke the graph on one image and return a serializable result.

    All vLLM concurrency is governed by the global ``_QWEN_SEM`` semaphore,
    so callers do not pass ``max_concurrency`` per-invocation.
    """
    initial: PipelineState = {
        "uuid": uuid, "source": source, "image_path": image_path,
        "tasks": [], "regions": [],
    }
    final = await graph.ainvoke(initial, config={"recursion_limit": 200})
    regions = sorted(final.get("regions", []),
                     key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return {"uuid": uuid, "source": source, "regions": regions}

## Demo on train samples

We run the pipeline end-to-end on a few examples from the public [`UkrainianCatholicUniversity/rukopys`](https://huggingface.co/datasets/UkrainianCatholicUniversity/rukopys/tree/main/train) HF dataset. The train split ships with ground-truth annotations (bbox + text), so we can compare predictions side-by-side with GT.

In [ ]:
# Download train metadata (contains GT for every page)
from huggingface_hub import hf_hub_download

REPO = "UkrainianCatholicUniversity/rukopys"

train_meta_path = hf_hub_download(
    repo_id=REPO, filename="train/metadata.jsonl", repo_type="dataset",
)

train_items = []
with open(train_meta_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_items.append(json.loads(line))
print(f"Train items in metadata: {len(train_items)}")

# Pick one example per source (school, dictation, university, archive)
demo_samples = {}
for it in train_items:
    src = it["source"]
    if src not in demo_samples:
        demo_samples[src] = it
demo_samples = list(demo_samples.values())
print(f"Selected {len(demo_samples)} demo samples (one per source):",
      [(s["source"], Path(s["file_name"]).stem[:8]) for s in demo_samples])

# Download just those four image files (full train split is ~GBs)
for s in demo_samples:
    s["local_path"] = hf_hub_download(
        repo_id=REPO, filename=f"train/{s['file_name']}", repo_type="dataset",
    )
print("Demo images downloaded.")


In [ ]:
# Run the pipeline on each demo sample concurrently via the async graph
init_concurrency(min(CONCURRENCY, len(demo_samples)))
demo_graph = build_graph()
demo_sem = asyncio.Semaphore(min(CONCURRENCY, len(demo_samples)))

async def _demo_one(s):
    async with demo_sem:
        uuid = Path(s["file_name"]).stem
        return await run_pipeline(demo_graph, s["local_path"], uuid, s["source"])

demo_results = await asyncio.gather(*(_demo_one(s) for s in demo_samples))
for s, r in zip(demo_samples, demo_results):
    print(f"  {s['source']:11s} {Path(s['file_name']).stem[:8]}: "
          f"GT={len(s.get('regions', []))} regions, predicted={len(r['regions'])}")


In [ ]:
# Side-by-side visualisation: ground truth (left) vs pipeline prediction (right)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# Bbox colour by region type
COLORS = {
    "handwritten": "#dc3232", "printed": "#3232dc", "formula":  "#ffa500",
    "table":       "#00c8c8", "image":   "#969696", "graph":    "#9632c8",
    "annotation":  "#c83296",
}


def _draw(ax, img, regions, title):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        c = COLORS.get(r.get("type", "handwritten"), "#666666")
        ax.add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.2, edgecolor=c, facecolor="none",
        ))


for s, r in zip(demo_samples, demo_results):
    img = Image.open(s["local_path"]).convert("RGB")
    fig, ax = plt.subplots(1, 2, figsize=(14, 9))

    _draw(ax[0], img, s.get("regions", []),
          f"Ground truth — {s['source']} ({len(s.get('regions', []))} regions)")

    pred_regions = [
        {"bbox": pr["bbox"], "type": pr.get("type", "handwritten"),
         "text": pr.get("transcription", "")}
        for pr in r["regions"]
    ]
    _draw(ax[1], img, pred_regions,
          f"Pipeline prediction ({len(pred_regions)} regions)")

    plt.tight_layout(); plt.show()

    # Text-level comparison — first 5 regions on each side
    print(f"\n=== {s['source']} / {Path(s['file_name']).stem[:8]} ===")
    print("GT:")
    for i, g in enumerate(s.get("regions", [])[:5], 1):
        print(f"  G{i} [{g.get('type','?')}]: {(g.get('text') or '')[:80]}")
    print("Predicted:")
    for i, p in enumerate(r["regions"][:5], 1):
        print(f"  P{i} [{p.get('type','?')}]: {(p.get('transcription') or '')[:80]}")
    print()


## Run inference on the test set

Build a manifest from `metadata.jsonl`, then run the async graph concurrently with `asyncio.gather`. We use top-level `await` (supported by the Jupyter/IPython kernel) — `asyncio.run` would fail because the kernel already owns an event loop.

In [ ]:
import asyncio
import json
from pathlib import Path
from tqdm.auto import tqdm

# Skip gracefully when the Kaggle competition data isn't mounted
# (e.g. running this notebook outside the competition environment)
if not os.path.exists(TEST_METADATA):
    print(f"{TEST_METADATA!r} not found — skipping the full-test-set run.")
    print("(Attach the competition dataset on Kaggle and re-execute this cell.)")
    items = []
else:
    items = []
    with open(TEST_METADATA) as f:
        for line in f:
            if not line.strip():
                continue
            m = json.loads(line)
            items.append({
                "uuid": Path(m["file_name"]).stem,
                "source": m["source"],
                "image_path": str(Path(TEST_IMAGES_DIR).parent / m["file_name"]),
            })
    print(f"Manifest: {len(items)} test images")

if items:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    init_concurrency(CONCURRENCY)
    graph = build_graph()
    sem_img = asyncio.Semaphore(CONCURRENCY)

    async def _process_one(item, pbar):
        async with sem_img:
            out_path = f"{OUTPUT_DIR}/{item['uuid']}.json"
            if os.path.exists(out_path):
                pbar.update(1); return
            try:
                result = await run_pipeline(graph, item["image_path"],
                                            item["uuid"], item["source"])
            except Exception as e:
                print(f"ERROR {item['uuid']}: {e}")
                result = {"uuid": item["uuid"], "source": item["source"], "regions": []}
            with open(out_path, "w") as f:
                json.dump(result, f, ensure_ascii=False)
            pbar.update(1)

    async def _run():
        pbar = tqdm(total=len(items))
        await asyncio.gather(*(_process_one(it, pbar) for it in items))
        pbar.close()

    await _run()


## Build submission.csv

Convert the per-image JSONs into a single CSV with columns `image, regions`, where `regions` is a JSON-encoded list of `{bbox, type, text}` objects. Failed transcriptions (anything starting with `[ERROR:`) are replaced with an empty string — they'll score more honestly as a miss than as garbage characters.

In [ ]:
import csv

def _clean(t: str) -> str:
    if not t or t.startswith("[ERROR:"):
        return ""
    return t

rows = []
if not os.path.isdir(OUTPUT_DIR) or not os.listdir(OUTPUT_DIR):
    print(f"{OUTPUT_DIR!r} is empty — run the inference cell first.")
else:
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if not fn.endswith(".json"):
            continue
        uuid = fn[:-5]
        with open(f"{OUTPUT_DIR}/{fn}") as f:
            data = json.load(f)
        regions = [
            {
                "bbox": r["bbox"],
                "type": r.get("type", "handwritten"),
                "text": _clean(r.get("transcription", "")),
            }
            for r in data.get("regions", [])
        ]
        rows.append({
            "image": f"{uuid}.jpg",
            "regions": json.dumps(regions, ensure_ascii=False),
        })

if rows:
    with open(SUBMISSION_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["image", "regions"])
        w.writeheader()
        w.writerows(rows)

print(f"Wrote {SUBMISSION_CSV}: {len(rows)} rows")


## Sanity-check submission

In [ ]:
import pandas as pd

if not os.path.exists(SUBMISSION_CSV):
    print(f"{SUBMISSION_CSV!r} not found — run the previous cells first.")
else:
    sub = pd.read_csv(SUBMISSION_CSV)
    print(f"Rows: {len(sub)}")
    print(f"Columns: {list(sub.columns)}")
    print(f"Sample (first 200 chars of regions): {sub.iloc[0]['regions'][:200]}")

    # Region count distribution
    import statistics
    counts = [len(json.loads(r)) for r in sub["regions"]]
    print(f"Regions/image: min={min(counts)}, median={statistics.median(counts):.0f}, "
          f"mean={statistics.mean(counts):.1f}, max={max(counts)}")


## What I'd improve next

| Effort | Change | Expected composite gain |
|---|---|---|
| ~30 min | NMS post-processing on line bboxes (drop pairs with IoU > 0.5) | +0.02 (kills duplicate-line FPs) |
| ~1 h | Few-shot examples added to the transcribe prompt | +0.02 – 0.04 (CER −3-5 %) |
| ~2 h | Auto-rotate skewed pages (fastdeskew / Hough) | +0.01 – 0.02 (better detect on tilted pages) |
| GPU + a few h | LoRA fine-tune Qwen3-VL-8B on the labelled train split (ms-swift) | +0.10 – 0.15 (Page CER is the top-weighted driver) |

For fine-tuning I'd recommend **ms-swift** (Alibaba's official framework with native Qwen3-VL grounding support), LoRA in bf16 with the ViT and aligner frozen, on a single H100 80 GB or A100 80 GB. Data goes in ms-swift's `messages` format with the `<image>` placeholder and **bboxes in the 0-1000 normalized scale** (matching the pipeline — resize-invariant, sidesteps Qwen3-VL's `smart_resize` coordinate drift).